
## Folder 04 — File 03: Silver Orders:


**Transformation rules:**

- Trim order_id, customer_id, currency, and sales_channel.
- Convert order_timestamp into a timestamp.
- Standardize order_status to uppercase.
- Convert order_total to DECIMAL(12,2).
- Quarantine missing IDs, invalid timestamps, unsupported statuses, missing currency, and negative totals.
- Deduplicate using order_id.
- Save valid records to silver_orders.


Allowed statuses in this dataset: **DELIVERED, SHIPPED, PROCESSING, CANCELLED**.

In [0]:

# ============================================================
# E-COMMERCE LAKEHOUSE - SILVER ORDERS TRANSFORMATION
# Standardization, validation, deduplication and quarantine
# ============================================================


## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"

BRONZE_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.bronze_orders"
)

SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_orders"
)

QUARANTINE_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.quarantine_orders"
)

ALLOWED_ORDER_STATUSES = [
    "DELIVERED",
    "SHIPPED",
    "PROCESSING",
    "CANCELLED"
]

print("Project configuration completed.")


## 3. Create Silver schema

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`{SILVER_SCHEMA}`"
)

print("Silver schema is ready.")


## 4. Read Bronze orders

if not spark.catalog.tableExists(BRONZE_TABLE):
    raise Exception(
        f"Bronze table not found: {BRONZE_TABLE}. "
        "Run the Bronze ingestion notebook first."
    )

bronze_df = spark.table(BRONZE_TABLE)

bronze_count = bronze_df.count()

print(f"BRONZE ORDER RECORDS: {bronze_count}")

display(bronze_df.limit(10))


## 5. Validate required source columns

required_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "currency",
    "sales_channel",
    "order_timestamp",
    "order_total"
]

missing_columns = [
    column
    for column in required_columns
    if column not in bronze_df.columns
]

if missing_columns:
    raise Exception(
        f"Required columns missing from Bronze: {missing_columns}"
    )


## 6. Clean and standardize order columns

cleaned_df = (
    bronze_df

    .withColumn(
        "order_id",
        F.trim(F.col("order_id").cast("string"))
    )

    .withColumn(
        "customer_id",
        F.trim(F.col("customer_id").cast("string"))
    )

    .withColumn(
        "order_status",
        F.upper(F.trim(F.col("order_status").cast("string")))
    )

    .withColumn(
        "currency",
        F.upper(F.trim(F.col("currency").cast("string")))
    )

    .withColumn(
        "sales_channel",
        F.initcap(F.trim(F.col("sales_channel").cast("string")))
    )

    .withColumn(
        "order_timestamp",
        F.expr(
            "try_cast(order_timestamp AS TIMESTAMP)"
        )
    )

    .withColumn(
        "order_total",
        F.expr(
            "try_cast(order_total AS DECIMAL(12,2))"
        )
    )
)

print("Order standardization completed.")

display(cleaned_df.limit(10))


## 7. Identify invalid order records

invalid_condition = (
    F.col("order_id").isNull()
    | (F.col("order_id") == "")

    | F.col("customer_id").isNull()
    | (F.col("customer_id") == "")

    | F.col("order_timestamp").isNull()

    | F.col("order_status").isNull()
    | (~F.col("order_status").isin(ALLOWED_ORDER_STATUSES))

    | F.col("currency").isNull()
    | (F.col("currency") == "")

    | F.col("order_total").isNull()
    | (F.col("order_total") < 0)
)

invalid_df = (
    cleaned_df
    .filter(invalid_condition)

    .withColumn(
        "_quarantine_reason",
        F.concat_ws(
            "; ",

            F.when(
                F.col("order_id").isNull()
                | (F.col("order_id") == ""),
                F.lit("Missing or empty order_id")
            ),

            F.when(
                F.col("customer_id").isNull()
                | (F.col("customer_id") == ""),
                F.lit("Missing or empty customer_id")
            ),

            F.when(
                F.col("order_timestamp").isNull(),
                F.lit("Missing or invalid order_timestamp")
            ),

            F.when(
                F.col("order_status").isNull()
                | (~F.col("order_status").isin(ALLOWED_ORDER_STATUSES)),
                F.lit("Invalid order_status")
            ),

            F.when(
                F.col("currency").isNull()
                | (F.col("currency") == ""),
                F.lit("Missing or empty currency")
            ),

            F.when(
                F.col("order_total").isNull(),
                F.lit("Missing or invalid order_total")
            ),

            F.when(
                F.col("order_total") < 0,
                F.lit("Negative order_total")
            )
        )
    )
)

valid_df = cleaned_df.filter(~invalid_condition)

valid_count = valid_df.count()
invalid_count = invalid_df.count()

print(f"VALID ORDER RECORDS: {valid_count}")
print(f"INVALID ORDER RECORDS: {invalid_count}")

print("QUARANTINE REASONS:")

display(
    invalid_df
    .groupBy("_quarantine_reason")
    .count()
    .orderBy(F.desc("count"))
)


## 8. Remove duplicate order records

# Use available ingestion metadata to select the latest
# record for each order_id.

preferred_order_columns = [
    "_ingestion_timestamp",
    "_ingested_at",
    "_source_file_modification_time",
    "_source_file"
]

order_columns = [
    column
    for column in preferred_order_columns
    if column in valid_df.columns
]

# Add business columns as deterministic tie-breakers.

for column in [
    "order_timestamp",
    "customer_id",
    "order_status",
    "order_total",
    "currency",
    "sales_channel"
]:
    if column in valid_df.columns and column not in order_columns:
        order_columns.append(column)

if not order_columns:
    raise Exception(
        "No columns available for deterministic deduplication."
    )

deduplication_window = (
    Window
    .partitionBy("order_id")
    .orderBy(
        *[
            F.col(column).desc_nulls_last()
            for column in order_columns
        ]
    )
)

ranked_df = valid_df.withColumn(
    "_row_number",
    F.row_number().over(deduplication_window)
)

# Keep one record per order_id.

deduplicated_df = (
    ranked_df
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

# Preserve duplicate records in quarantine.

duplicate_df = (
    ranked_df
    .filter(F.col("_row_number") > 1)
    .drop("_row_number")

    .withColumn(
        "_quarantine_reason",
        F.lit("Duplicate order_id")
    )
)

deduplicated_count = deduplicated_df.count()
duplicate_count = duplicate_df.count()

print(f"DEDUPLICATED SILVER RECORDS: {deduplicated_count}")
print(f"DUPLICATE RECORDS: {duplicate_count}")


## 9. Combine invalid and duplicate records

quarantine_df = (
    invalid_df
    .unionByName(
        duplicate_df,
        allowMissingColumns=True
    )
    .withColumn(
        "quarantined_at",
        F.current_timestamp()
    )
)

quarantine_count = quarantine_df.count()

print(f"TOTAL QUARANTINE RECORDS: {quarantine_count}")

display(
    quarantine_df
    .groupBy("_quarantine_reason")
    .count()
    .orderBy(F.desc("count"))
)


## 10. Save invalid and duplicate records to quarantine

(
    quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table written: {QUARANTINE_TABLE}")


## 11. Save clean records to Silver Delta table

(
    deduplicated_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table written: {SILVER_TABLE}")


## 12. Validate Silver order records

silver_df = spark.table(SILVER_TABLE)

total_silver_records = silver_df.count()

duplicate_order_groups = (
    silver_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_order_records = (
    silver_df
    .filter(
        F.col("order_id").isNull()
        | (F.col("order_id") == "")

        | F.col("customer_id").isNull()
        | (F.col("customer_id") == "")

        | F.col("order_timestamp").isNull()

        | F.col("order_status").isNull()
        | (~F.col("order_status").isin(ALLOWED_ORDER_STATUSES))

        | F.col("currency").isNull()
        | (F.col("currency") == "")

        | F.col("order_total").isNull()
        | (F.col("order_total") < 0)
    )
    .count()
)

print("=" * 55)
print("SILVER ORDER VALIDATION")
print("=" * 55)

print(f"Silver records: {total_silver_records}")
print(f"Duplicate order groups: {duplicate_order_groups}")
print(f"Invalid order records: {invalid_order_records}")

if (
    duplicate_order_groups > 0
    or invalid_order_records > 0
):
    raise Exception("Silver order validation failed.")

print("Silver order validation passed.")


## 13. Analyze order statuses

status_summary_df = (
    silver_df
    .groupBy("order_status")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum("order_total").alias("total_order_value")
    )
    .orderBy("order_status")
)

display(status_summary_df)


## 14. Bronze to Silver reconciliation

bronze_distinct_ids = (
    cleaned_df
    .filter(
        F.col("order_id").isNotNull()
        & (F.col("order_id") != "")
    )
    .select("order_id")
    .distinct()
)

silver_distinct_ids = (
    silver_df
    .select("order_id")
    .distinct()
)

missing_from_silver = (
    bronze_distinct_ids
    .join(
        silver_distinct_ids,
        on="order_id",
        how="left_anti"
    )
)

missing_silver_count = missing_from_silver.count()

print("=" * 55)
print("BRONZE TO SILVER RECONCILIATION")
print("=" * 55)

print(f"Bronze total rows: {bronze_count}")

print(
    f"Bronze distinct order IDs: "
    f"{bronze_distinct_ids.count()}"
)

print(f"Silver total rows: {total_silver_records}")

print(
    f"Silver distinct order IDs: "
    f"{silver_distinct_ids.count()}"
)

print(
    f"Unique Bronze order IDs missing from Silver: "
    f"{missing_silver_count}"
)

display(missing_from_silver.limit(20))


## 15. Check quarantine explanation for missing order IDs

quarantine_ids = (
    quarantine_df
    .filter(F.col("order_id").isNotNull())
    .select("order_id")
    .distinct()
)

unexplained_missing_ids = (
    missing_from_silver
    .join(
        quarantine_ids,
        on="order_id",
        how="left_anti"
    )
)

unexplained_count = unexplained_missing_ids.count()

print(
    f"Missing order IDs without quarantine explanation: "
    f"{unexplained_count}"
)

display(unexplained_missing_ids.limit(20))


## 16. Customer reference check - informational only

# Do NOT quarantine otherwise-valid orders just because
# their customer_id is absent from Silver.
# The separate business-rule validation handles this.

CUSTOMER_SILVER_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_customers"
)

if spark.catalog.tableExists(CUSTOMER_SILVER_TABLE):

    customer_ids = (
        spark.table(CUSTOMER_SILVER_TABLE)
        .select("customer_id")
        .distinct()
    )

    orders_without_customer = (
        silver_df
        .select("order_id", "customer_id")
        .join(
            customer_ids,
            on="customer_id",
            how="left_anti"
        )
    )

    missing_customer_reference_count = (
        orders_without_customer.count()
    )

    print("=" * 55)
    print("CUSTOMER REFERENCE CHECK")
    print("=" * 55)

    print(
        "Orders referencing customers absent from Silver: "
        f"{missing_customer_reference_count}"
    )

    display(orders_without_customer.limit(20))

else:
    print(
        "Customer Silver table not found. "
        "Skipping informational reference check."
    )


## 17. Final summary

print("=" * 55)
print("ORDER SILVER TRANSFORMATION COMPLETED")
print("=" * 55)

print(f"Bronze table: {BRONZE_TABLE}")
print(f"Silver table: {SILVER_TABLE}")
print(f"Quarantine table: {QUARANTINE_TABLE}")

print(f"Bronze records: {bronze_count}")
print(f"Valid records before deduplication: {valid_count}")
print(f"Invalid records: {invalid_count}")
print(f"Duplicate records quarantined: {duplicate_count}")
print(f"Silver unique orders: {total_silver_records}")
print(f"Total quarantine records: {quarantine_count}")
print(f"Unaccounted missing order IDs: {unexplained_count}")

if unexplained_count > 0:
    raise Exception(
        "Some missing order IDs have no quarantine explanation."
    )

print("=" * 55)
print("FINAL SILVER ORDER VALIDATION PASSED")
print("=" * 55)

display(
    silver_df
    .orderBy("order_timestamp")
    .limit(20)
)
